# Lab 5


Matrix Representation: In this lab you will be creating a simple linear algebra system. In memory, we will represent matrices as nested python lists as we have done in lecture. In the exercises below, you are required to explicitly test every feature you implement, demonstrating it works.

1. Create a `matrix` class with the following properties:
    * It can be initialized in 2 ways:
        1. with arguments `n` and `m`, the size of the matrix. A newly instanciated matrix will contain all zeros.
        2. with a list of lists of values. Note that since we are using lists of lists to implement matrices, it is possible that not all rows have the same number of columns. Test explicitly that the matrix is properly specified.
    * Matrix instances `M` can be indexed with `M[i][j]` and `M[i,j]`.
    * Matrix assignment works in 2 ways:
        1. If `M_1` and `M_2` are `matrix` instances `M_1=M_2` sets the values of `M_1` to those of `M_2`, if they are the same size. Error otherwise.
        2. In example above `M_2` can be a list of lists of correct size.


In [1]:
class Matrix:
    def __init__(self, n=None, m=None, values=None):
        if values is not None:
            if any(len(row) != len(values[0]) for row in values):
                raise ValueError("All rows must have same length")
            self.data = [row[:] for row in values]
        elif n is not None and m is not None:
            self.data = [[0.0]*m for _ in range(n)]
        else:
            raise ValueError("Provide size or list of lists")
        self.n = len(self.data)
        self.m = len(self.data[0])

    def __getitem__(self, key):
        return self.data[key[0]][key[1]] if isinstance(key, tuple) else self.data[key]

    def __setitem__(self, key, value):
        if isinstance(key, tuple):
            self.data[key[0]][key[1]] = value
        else:
            self.data[key] = value

    def assign(self, other):
        if isinstance(other, Matrix):
            if self.n != other.n or self.m != other.m:
                raise ValueError("Sizes do not match")
            self.data = [row[:] for row in other.data]
        elif isinstance(other, list):
            if len(other) != self.n or any(len(row) != self.m for row in other):
                raise ValueError("List size mismatch")
            self.data = [row[:] for row in other]

    def __repr__(self):
        return str(self.data)

In [2]:
M1 = Matrix(2,3)
M2 = Matrix(values=[[1,2,3],[4,5,6]])
M2[0,1] = 20
print(M2[0][2])
M1.assign(M2)
print(M1)

3
[[1, 20, 3], [4, 5, 6]]


2. Add the following methods:
    * `shape()`: returns a tuple `(n,m)` of the shape of the matrix.
    * `transpose()`: returns a new matrix instance which is the transpose of the matrix.
    * `row(n)` and `column(n)`: that return the nth row or column of the matrix M as a new appropriately shaped matrix object.
    * `to_list()`: which returns the matrix as a list of lists.
    *  `block(n_0,n_1,m_0,m_1)` that returns a smaller matrix located at the n_0 to n_1 columns and m_0 to m_1 rows. 
    * Modify `__getitem__` implemented above to support slicing.
        

In [4]:
class Matrix:
    def __init__(self, n=None, m=None, values=None):
        if values is not None:
            if any(len(row) != len(values[0]) for row in values):
                raise ValueError("All rows must have same length")
            self.data = [row[:] for row in values]
        elif n is not None and m is not None:
            self.data = [[0.0]*m for _ in range(n)]
        else:
            raise ValueError("Provide size or list of lists")
        self.n = len(self.data)
        self.m = len(self.data[0])

    def __getitem__(self, key):
        if isinstance(key, tuple):
            row, col = key
            if isinstance(row, slice) or isinstance(col, slice):
                rows = self.data[row] if isinstance(row, slice) else [self.data[row]]
                new_data = [r[col] if isinstance(col, slice) else [r[col]] for r in rows]
                return Matrix(values=new_data)
            else:
                return self.data[row][col]
        else:
            return self.data[key]

    def __setitem__(self, key, value):
        if isinstance(key, tuple):
            self.data[key[0]][key[1]] = value
        else:
            self.data[key] = value

    def assign(self, other):
        if isinstance(other, Matrix):
            if self.n != other.n or self.m != other.m:
                raise ValueError("Sizes do not match")
            self.data = [row[:] for row in other.data]
        elif isinstance(other, list):
            if len(other) != self.n or any(len(row) != self.m for row in other):
                raise ValueError("List size mismatch")
            self.data = [row[:] for row in other]

    def shape(self):
        return (self.n, self.m)

    def transpose(self):
        new_data = [[self.data[i][j] for i in range(self.n)] for j in range(self.m)]
        return Matrix(values=new_data)

    def row(self, n):
        return Matrix(values=[self.data[n]])

    def column(self, n):
        return Matrix(values=[[self.data[i][n]] for i in range(self.n)])

    def to_list(self):
        return [row[:] for row in self.data]

    def block(self, n0, n1, m0, m1):
        new_data = [row[m0:m1] for row in self.data[n0:n1]]
        return Matrix(values=new_data)

    def __repr__(self):
        return str(self.data)

In [6]:
M = Matrix(values=[[1,2,3],[4,5,6],[7,8,9]])
print("Shape:", M.shape())

T = M.transpose()
print("Transpose:\n", T)
print("Row 1:", M.row(1))
print("Column 2:", M.column(2))
print("Block 0:2,1:3:", M.block(0,2,1,3))
print("To list:", M.to_list())
print("Slice rows 0:2, cols 1:3:", M[0:2,1:3])
print("Single element:", M[1,2])

Shape: (3, 3)
Transpose:
 [[1, 4, 7], [2, 5, 8], [3, 6, 9]]
Row 1: [[4, 5, 6]]
Column 2: [[3], [6], [9]]
Block 0:2,1:3: [[2, 3], [5, 6]]
To list: [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
Slice rows 0:2, cols 1:3: [[2, 3], [5, 6]]
Single element: 6


3. Write functions that create special matrices (note these are standalone functions, not member functions of your `matrix` class):
    * `constant(n,m,c)`: returns a `n` by `m` matrix filled with floats of value `c`.
    * `zeros(n,m)` and `ones(n,m)`: return `n` by `m` matrices filled with floats of value `0` and `1`, respectively.
    * `eye(n)`: returns the n by n identity matrix.

In [9]:
def constant(n, m, c):
    return Matrix(values=[[float(c) for _ in range(m)] for _ in range(n)])

def zeros(n, m):
    return constant(n, m, 0)

def ones(n, m):
    return constant(n, m, 1)

def eye(n):
    return Matrix(values=[[1.0 if i == j else 0.0 for j in range(n)] for i in range(n)])

In [11]:
print("2x3:", constant(2,3,7))
print("3x2:", zeros(3,2))
print("2x4:", ones(2,4))
print("4x4:", eye(4))

2x3: [[7.0, 7.0, 7.0], [7.0, 7.0, 7.0]]
3x2: [[0.0, 0.0], [0.0, 0.0], [0.0, 0.0]]
2x4: [[1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0]]
4x4: [[1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0], [0.0, 0.0, 1.0, 0.0], [0.0, 0.0, 0.0, 1.0]]


4. Add the following member functions to your class. Make sure to appropriately test the dimensions of the matrices to make sure the operations are correct.
    * `M.scalarmul(c)`: a matrix that is scalar product $cM$, where every element of $M$ is multiplied by $c$.
    * `M.add(N)`: adds two matrices $M$ and $N$. Don’t forget to test that the sizes of the matrices are compatible for this and all other operations.
    * `M.sub(N)`: subtracts two matrices $M$ and $N$.
    * `M.mat_mult(N)`: returns a matrix that is the matrix product of two matrices $M$ and $N$.
    * `M.element_mult(N)`: returns a matrix that is the element-wise product of two matrices $M$ and $N$.
    * `M.equals(N)`: returns true/false if $M==N$.

In [13]:
class Matrix:
    def __init__(self, n=None, m=None, values=None):
        if values:
            if any(len(row) != len(values[0]) for row in values):
                raise ValueError("All rows must have same length")
            self.data = [row[:] for row in values]
        elif n is not None and m is not None:
            self.data = [[0.0]*m for _ in range(n)]
        else:
            raise ValueError("Provide size or values")
        self.n, self.m = len(self.data), len(self.data[0])

    def __getitem__(self, key):
        if isinstance(key, tuple):
            r, c = key
            rows = self.data[r] if isinstance(r, slice) else [self.data[r]]
            result = [row[c] if isinstance(c, slice) else [row[c]] for row in rows]
            return Matrix(values=result) if isinstance(r, slice) or isinstance(c, slice) else result[0][0]
        return self.data[key]

    def __setitem__(self, key, value):
        if isinstance(key, tuple): self.data[key[0]][key[1]] = value
        else: self.data[key] = value

    def shape(self): return (self.n, self.m)
    def to_list(self): return [row[:] for row in self.data]
    def transpose(self): return Matrix(values=[[self.data[i][j] for i in range(self.n)] for j in range(self.m)])
    def row(self, i): return Matrix(values=[self.data[i]])
    def column(self, j): return Matrix(values=[[self.data[i][j]] for i in range(self.n)])
    def block(self, r0, r1, c0, c1): return Matrix(values=[row[c0:c1] for row in self.data[r0:r1]])

    def scalarmul(self, c): return Matrix(values=[[x*c for x in row] for row in self.data])
    def add(self, B): return Matrix(values=[[self.data[i][j]+B.data[i][j] for j in range(self.m)] for i in range(self.n)])
    def sub(self, B): return Matrix(values=[[self.data[i][j]-B.data[i][j] for j in range(self.m)] for i in range(self.n)])
    def element_mult(self, B): return Matrix(values=[[self.data[i][j]*B.data[i][j] for j in range(self.m)] for i in range(self.n)])
    def mat_mult(self, B): return Matrix(values=[[sum(self.data[i][k]*B.data[k][j] for k in range(self.m)) for j in range(B.m)] for i in range(self.n)])
    def equals(self, B): return self.data == B.data

    def __repr__(self): return str(self.data)

In [15]:
A = Matrix(values=[[1,2],[3,4]])
B = Matrix(values=[[5,6],[7,8]])

print("A:", A)
print("B:", B)
print("2*A:", A.scalarmul(2))
print("A+B:", A.add(B))
print("A-B:", A.sub(B))
print("A*B:", A.mat_mult(B))
print("A element-wise B:", A.element_mult(B))
print("A == B?", A.equals(B))
print("A transpose:", A.transpose())
print("Row 1:", A.row(1))
print("Column 0:", A.column(0))
print("Block 0:2,0:1:", A.block(0,2,0,1))

A: [[1, 2], [3, 4]]
B: [[5, 6], [7, 8]]
2*A: [[2, 4], [6, 8]]
A+B: [[6, 8], [10, 12]]
A-B: [[-4, -4], [-4, -4]]
A*B: [[19, 22], [43, 50]]
A element-wise B: [[5, 12], [21, 32]]
A == B? False
A transpose: [[1, 3], [2, 4]]
Row 1: [[3, 4]]
Column 0: [[1], [3]]
Block 0:2,0:1: [[1], [3]]


5. Overload python operators to appropriately use your functions in 4 and allow expressions like:
    * 2*M
    * M*2
    * M+N
    * M-N
    * M*N
    * M==N
    * M=N


In [18]:
class Matrix:
    def __init__(self, n=None, m=None, values=None):
        if values:
            if any(len(row) != len(values[0]) for row in values): raise ValueError("Rows must match")
            self.data = [row[:] for row in values]
        elif n is not None and m is not None:
            self.data = [[0.0]*m for _ in range(n)]
        else: raise ValueError("Provide size or values")
        self.n, self.m = len(self.data), len(self.data[0])

    def __getitem__(self, key):
        if isinstance(key, tuple):
            r, c = key
            rows = self.data[r] if isinstance(r, slice) else [self.data[r]]
            result = [row[c] if isinstance(c, slice) else [row[c]] for row in rows]
            return Matrix(values=result) if isinstance(r, slice) or isinstance(c, slice) else result[0][0]
        return self.data[key]

    def __setitem__(self, key, value):
        self.data[key[0]][key[1]] = value if isinstance(key, tuple) else self.data.__setitem__(key,value)

    def shape(self): return (self.n, self.m)
    def to_list(self): return [row[:] for row in self.data]
    def transpose(self): return Matrix(values=[[self.data[i][j] for i in range(self.n)] for j in range(self.m)])
    def row(self, i): return Matrix(values=[self.data[i]])
    def column(self, j): return Matrix(values=[[self.data[i][j]] for i in range(self.n)])
    def block(self,r0,r1,c0,c1): return Matrix(values=[row[c0:c1] for row in self.data[r0:r1]])

    def scalarmul(self,c): return Matrix(values=[[x*c for x in row] for row in self.data])
    def add(self,B): return Matrix(values=[[self.data[i][j]+B.data[i][j] for j in range(self.m)] for i in range(self.n)])
    def sub(self,B): return Matrix(values=[[self.data[i][j]-B.data[i][j] for j in range(self.m)] for i in range(self.n)])
    def element_mult(self,B): return Matrix(values=[[self.data[i][j]*B.data[i][j] for j in range(self.m)] for i in range(self.n)])
    def mat_mult(self,B): return Matrix(values=[[sum(self.data[i][k]*B.data[k][j] for k in range(self.m)) for j in range(B.m)] for i in range(self.n)])
    def equals(self,B): return self.data==B.data

    __add__ = add
    __sub__ = sub
    __mul__ = lambda self, other: self.scalarmul(other) if isinstance(other,(int,float)) else self.mat_mult(other)
    __rmul__ = lambda self, other: self.scalarmul(other)
    __eq__ = equals
    def __repr__(self): return str(self.data)

In [21]:
A = Matrix(values=[[1,2],[3,4]])
B = Matrix(values=[[5,6],[7,8]])

print("A + B =", A+B)
print("A - B =", A-B)
print("A * B =", A*B)
print("2 * A =", 2*A)
print("A * 3 =", A*3)
print("A transpose =", A.transpose())
print("Row 1 =", A.row(1))
print("Column 0 =", A.column(0))
print("Block 0:2,0:1 =", A.block(0,2,0,1))
print("A == B", A==B)

A + B = [[6, 8], [10, 12]]
A - B = [[-4, -4], [-4, -4]]
A * B = [[19, 22], [43, 50]]
2 * A = [[2, 4], [6, 8]]
A * 3 = [[3, 6], [9, 12]]
A transpose = [[1, 3], [2, 4]]
Row 1 = [[3, 4]]
Column 0 = [[1], [3]]
Block 0:2,0:1 = [[1], [3]]
A == B False


6. Demonstrate the basic properties of matrices with your matrix class by creating two 2 by 2 example matrices using your Matrix class and illustrating the following:

$$
(AB)C=A(BC)
$$
$$
A(B+C)=AB+AC
$$
$$
AB\neq BA
$$
$$
AI=A
$$

In [27]:
A = Matrix(values=[[1, 2],[3, 4]])
B = Matrix(values=[[5, 6],[7, 8]])
C = Matrix(values=[[2, 0],[1, 2]])
I = Matrix(values=[[1, 0],[0, 1]])

ABC1 = (A * B) * C
ABC2 = A * (B * C)
print("((AB)C = A(BC)):")
print(" (AB)C =", ABC1)
print(" A(BC) =", ABC2)
print(" Equal?", ABC1 == ABC2)
print()

BC_sum = B + C
AB_AC = (A * B) + (A * C)
print("(A(B+C) = AB + AC):")
print(" A(B+C) =", A * BC_sum)
print(" AB + AC =", AB_AC)
print(" Equal?", (A * BC_sum) == AB_AC)
print()

AB = A * B
BA = B * A
print("(AB != BA):")
print(" AB =", AB)
print(" BA =", BA)
print(" AB == BA?", AB == BA)
print()

AI = A * I
print("(AI = A):")
print(" AI =", AI)
print(" Equal to A?", AI == A)

((AB)C = A(BC)):
 (AB)C = [[60, 44], [136, 100]]
 A(BC) = [[60, 44], [136, 100]]
 Equal? True

(A(B+C) = AB + AC):
 A(B+C) = [[23, 26], [53, 58]]
 AB + AC = [[23, 26], [53, 58]]
 Equal? True

(AB != BA):
 AB = [[19, 22], [43, 50]]
 BA = [[23, 34], [31, 46]]
 AB == BA? False

(AI = A):
 AI = [[1, 2], [3, 4]]
 Equal to A? True
